In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import util.trec_io
from pref_eval import get_measures, get_prefs
import glob
from tqdm.notebook import tqdm

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
dataset = "ml-1M"
qrels_file = f"../../data-source/qrels/{dataset}/2018.txt"
runfiles = glob.glob(f"../../data-source/runs/{dataset}/2018/*.gz")

In [15]:
measures = get_measures(["rpp", "invrpp", "dcgrpp", "ap", "ndcg", "rr"], None)
print(measures)

['rpp', 'invrpp', 'dcgrpp', 'ap', 'ndcg', 'rr']


In [16]:
qrels = util.trec_io.read_qrels(qrels_file, 4, None)

In [17]:
runs = {}
for runfile in tqdm(runfiles):
    runid, run = util.trec_io.read_run(runfile, qrels, None)
    runs[runid] = run

  0%|          | 0/21 [00:00<?, ?it/s]

In [31]:
summary = get_prefs(1, runs, measures, True)

  0%|          | 0/6005 [00:00<?, ?it/s]

In [21]:
df = pd.read_json("preferences.jsonl", lines=True)

In [24]:
groups = df.groupby(["runi", "runj"])

In [29]:
from scipy.stats import ttest_1samp

results = []

metrics = ["rpp", "invrpp", "dcgrpp", "ap", "ndcg", "rr"]

for (runi, runj), g in groups:
    for m in metrics:
        vals = g[m]

        if len(vals) > 1:
            tstat, pval = ttest_1samp(vals, 0.0)

            results.append(
                {
                    "runi": runi,
                    "runj": runj,
                    "metric": m,
                    "tstat": tstat,
                    "pval": pval,
                    "n": len(vals),
                }
            )

res_df = pd.DataFrame(results)

In [30]:
res_df

,runi,runj,metric,tstat,pval,n
0,BPRMF,HT,rpp,194.071827,0.0,6005
1,BPRMF,HT,invrpp,277.965950,0.0,6005
2,BPRMF,HT,dcgrpp,237.738772,0.0,6005
3,BPRMF,HT,ap,109.903549,0.0,6005
4,BPRMF,HT,ndcg,170.385339,0.0,6005
...,...,...,...,...,...,...
1255,WRMF,SVD,invrpp,184.508466,0.0,6005
1256,WRMF,SVD,dcgrpp,178.407294,0.0,6005
1257,WRMF,SVD,ap,100.928069,0.0,6005
1258,WRMF,SVD,ndcg,141.181670,0.0,6005


In [26]:
from statsmodels.stats.multitest import multipletests

reject, pvals_corrected, _, _ = multipletests(res_df["pval"], method="bonferroni")

res_df["pval_corrected"] = pvals_corrected
res_df["significant"] = reject

In [27]:
def get_percentage(res_df, metric: str):
    return len(res_df[(res_df["metric"] == metric) & (res_df["significant"])]) / 210

In [28]:
for metric in metrics:
    print(f"{metric}: {get_percentage(res_df, metric) * 100:.2f}")

rpp: 95.24
invrpp: 95.71
dcgrpp: 95.24
ap: 93.33
ndcg: 95.24
rr: 86.19
